In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TitanicCSV") \
    .getOrCreate()

df = spark.read.csv("titanic.csv", header=True, inferSchema=True)

df.show(10)

+--------+------+------+----+-----+-----+-------+--------+----+
|survived|pclass|gender| age|sibsp|parch|   fare|embarked|deck|
+--------+------+------+----+-----+-----+-------+--------+----+
|       0|     3|  male|22.0|    1|    0|   7.25|       S|NULL|
|       1|     1|female|38.0|    1|    0|71.2833|       C|   C|
|       1|     3|female|26.0|    0|    0|  7.925|       S|NULL|
|       1|     1|female|35.0|    1|    0|   53.1|       S|   C|
|       0|     3|  male|35.0|    0|    0|   8.05|       S|NULL|
|       0|     3|  male|NULL|    0|    0| 8.4583|       Q|NULL|
|       0|     1|  male|54.0|    0|    0|51.8625|       S|   E|
|       0|     3|  male| 2.0|    3|    1| 21.075|       S|NULL|
|       1|     3|female|27.0|    0|    2|11.1333|       S|NULL|
|       1|     2|female|14.0|    1|    0|30.0708|       C|NULL|
+--------+------+------+----+-----+-----+-------+--------+----+
only showing top 10 rows


In [3]:
# Display schema
df.printSchema()

root
 |-- survived: integer (nullable = true)
 |-- pclass: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: double (nullable = true)
 |-- sibsp: integer (nullable = true)
 |-- parch: integer (nullable = true)
 |-- fare: double (nullable = true)
 |-- embarked: string (nullable = true)
 |-- deck: string (nullable = true)



In [4]:
# Display first 10 records
df.show(10)

+--------+------+------+----+-----+-----+-------+--------+----+
|survived|pclass|gender| age|sibsp|parch|   fare|embarked|deck|
+--------+------+------+----+-----+-----+-------+--------+----+
|       0|     3|  male|22.0|    1|    0|   7.25|       S|NULL|
|       1|     1|female|38.0|    1|    0|71.2833|       C|   C|
|       1|     3|female|26.0|    0|    0|  7.925|       S|NULL|
|       1|     1|female|35.0|    1|    0|   53.1|       S|   C|
|       0|     3|  male|35.0|    0|    0|   8.05|       S|NULL|
|       0|     3|  male|NULL|    0|    0| 8.4583|       Q|NULL|
|       0|     1|  male|54.0|    0|    0|51.8625|       S|   E|
|       0|     3|  male| 2.0|    3|    1| 21.075|       S|NULL|
|       1|     3|female|27.0|    0|    2|11.1333|       S|NULL|
|       1|     2|female|14.0|    1|    0|30.0708|       C|NULL|
+--------+------+------+----+-----+-----+-------+--------+----+
only showing top 10 rows


In [5]:
# remove rows having null values in age column.

df = df.filter(df.age.isNotNull())
df.show(10)


+--------+------+------+----+-----+-----+-------+--------+----+
|survived|pclass|gender| age|sibsp|parch|   fare|embarked|deck|
+--------+------+------+----+-----+-----+-------+--------+----+
|       0|     3|  male|22.0|    1|    0|   7.25|       S|NULL|
|       1|     1|female|38.0|    1|    0|71.2833|       C|   C|
|       1|     3|female|26.0|    0|    0|  7.925|       S|NULL|
|       1|     1|female|35.0|    1|    0|   53.1|       S|   C|
|       0|     3|  male|35.0|    0|    0|   8.05|       S|NULL|
|       0|     1|  male|54.0|    0|    0|51.8625|       S|   E|
|       0|     3|  male| 2.0|    3|    1| 21.075|       S|NULL|
|       1|     3|female|27.0|    0|    2|11.1333|       S|NULL|
|       1|     2|female|14.0|    1|    0|30.0708|       C|NULL|
|       1|     3|female| 4.0|    1|    1|   16.7|       S|   G|
+--------+------+------+----+-----+-----+-------+--------+----+
only showing top 10 rows


In [6]:
from pyspark.sql import functions as F

df = df.withColumn(
    "Age_Group",
    F.when(F.col("age") < 18, "Child")
    .when((F.col("age") >= 18) & (F.col("age") <= 59), "Adult")
    .otherwise("Senior")
)
df.show(10)

+--------+------+------+----+-----+-----+-------+--------+----+---------+
|survived|pclass|gender| age|sibsp|parch|   fare|embarked|deck|Age_Group|
+--------+------+------+----+-----+-----+-------+--------+----+---------+
|       0|     3|  male|22.0|    1|    0|   7.25|       S|NULL|    Adult|
|       1|     1|female|38.0|    1|    0|71.2833|       C|   C|    Adult|
|       1|     3|female|26.0|    0|    0|  7.925|       S|NULL|    Adult|
|       1|     1|female|35.0|    1|    0|   53.1|       S|   C|    Adult|
|       0|     3|  male|35.0|    0|    0|   8.05|       S|NULL|    Adult|
|       0|     1|  male|54.0|    0|    0|51.8625|       S|   E|    Adult|
|       0|     3|  male| 2.0|    3|    1| 21.075|       S|NULL|    Child|
|       1|     3|female|27.0|    0|    2|11.1333|       S|NULL|    Adult|
|       1|     2|female|14.0|    1|    0|30.0708|       C|NULL|    Child|
|       1|     3|female| 4.0|    1|    1|   16.7|       S|   G|    Child|
+--------+------+------+----+-----+---

In [7]:
# Register temp view
df.createOrReplaceTempView("titanic")

In [8]:
# Find number of passengers in each passenger class

spark.sql("""
    SELECT pclass, COUNT(*) AS passenger_count
    FROM titanic
    GROUP BY pclass
""").show()


+------+---------------+
|pclass|passenger_count|
+------+---------------+
|     1|            186|
|     3|            355|
|     2|            173|
+------+---------------+



In [9]:
# calculate the avg age for each passenger class

spark.sql("""
    SELECT pclass, AVG(age) AS avg_age
    FROM titanic
    GROUP BY pclass
""").show()

+------+------------------+
|pclass|           avg_age|
+------+------------------+
|     1|38.233440860215055|
|     3| 25.14061971830986|
|     2| 29.87763005780347|
+------+------------------+



In [10]:
# display top 5 oldest passengers

spark.sql("""
    SELECT *
    FROM titanic
    ORDER BY age DESC
    LIMIT 5
""").show()

+--------+------+------+----+-----+-----+-------+--------+----+---------+
|survived|pclass|gender| age|sibsp|parch|   fare|embarked|deck|Age_Group|
+--------+------+------+----+-----+-----+-------+--------+----+---------+
|       1|     1|  male|80.0|    0|    0|   30.0|       S|   A|   Senior|
|       0|     3|  male|74.0|    0|    0|  7.775|       S|NULL|   Senior|
|       0|     1|  male|71.0|    0|    0|34.6542|       C|   A|   Senior|
|       0|     1|  male|71.0|    0|    0|49.5042|       C|NULL|   Senior|
|       0|     3|  male|70.5|    0|    0|   7.75|       Q|NULL|   Senior|
+--------+------+------+----+-----+-----+-------+--------+----+---------+



In [13]:
# save the transformed Dataframe as a csv file in new directory

df.write.csv("titanic_transformed.csv", header=True, mode="overwrite")